# Лабораторная 5: персональная text2img (Textual Inversion)

**Задача:** дообучить предобученную text2img-модель на своих фото, сохранить портретное сходство и сгенерировать изображения в разных окружениях.

**Подход:** [Stable Diffusion 1.5](https://huggingface.co/runwayml/stable-diffusion-v1-5) + **Textual Inversion** (как в эталонном `main.ipynb`). Обучается только эмбеддинг нового токена в CLIP — UNet и VAE **не меняются** → модель не «забывает» других людей.

**Почему не LoRA:** на MPS LoRA+diffusers часто даёт чёрные кадры; TI стабильнее и проверен эталоном.

**Ограничения:** только text2img, без negative prompt, без API — всё локально через `diffusers`.

**Подготовка:** 4–8 фото в `data/my_photos/`, задайте `PLACEHOLDER_TOKEN` и `GENDER`.

**Запуск:** cwd = `lab5/`, Jupyter Notebook 7.


## 0. Зависимости


In [ ]:
# %pip install -r requirements.txt


## 1. Конфигурация


In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

LAB_ROOT = Path.cwd()
PHOTOS_DIR = LAB_ROOT / "data" / "my_photos"
FACES_DIR = LAB_ROOT / "data" / "my_photos_faces"
CHECKPOINT_DIR = LAB_ROOT / "checkpoints"
OUTPUT_DIR = LAB_ROOT / "outputs"
for p in (PHOTOS_DIR, FACES_DIR, CHECKPOINT_DIR, OUTPUT_DIR):
    p.mkdir(parents=True, exist_ok=True)


def pick_device() -> str:
    forced = os.environ.get("DEVICE")
    if forced:
        return forced
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


DEVICE = pick_device()
WEIGHT_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
SKIP_TRAIN = os.environ.get("SKIP_TRAIN", "0") == "1"

# === НАСТРОЙТЕ ПОД СЕБЯ ===
PLACEHOLDER_TOKEN = os.environ.get("PLACEHOLDER_TOKEN", "<mytoken>")
GENDER = os.environ.get("GENDER", "man")  # для промптов без токена
INITIALIZER_TOKEN = os.environ.get("INITIALIZER_TOKEN", GENDER)  # инициализация эмбеддинга

MODEL_ID = "runwayml/stable-diffusion-v1-5"
EMBEDS_FILE = CHECKPOINT_DIR / "learned_embeds.pt"

TRAIN_TEMPLATES = [
    "a portrait photo of a {}, looking at the camera",
    "a portrait of a {}, looking at the camera",
    "a close-up portrait of a {}, looking at the camera",
    "a headshot of a {}, looking at the camera",
    "a face photo of a {}, looking at the camera",
    "a photo of a {}, looking at the camera",
    "a high quality portrait of a {}, looking at the camera",
    "a rendering of a {}, facing the viewer",
    "a bright photo of a {} looking at the camera",
    "a photo of a {} facing forward, well lit face",
]


@dataclass
class Config:
    img_size: int = 512
    face_fraction: float = 0.65
    batch_size: int = 1
    learning_rate: float = float(os.environ.get("LEARNING_RATE", "5e-4"))
    train_steps: int = int(os.environ.get("TRAIN_STEPS", "2000"))
    seed: int = 42
    num_inference_steps: int = 50
    guidance_scale: float = 7.5


CFG = Config()
LOSS_HISTORY_PATH = OUTPUT_DIR / "training_loss.json"

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)

print(f"Device: {DEVICE}, dtype: {WEIGHT_DTYPE}")
print(f"Token: {PLACEHOLDER_TOKEN!r}, gender: {GENDER!r}, init: {INITIALIZER_TOKEN!r}")
print(asdict(CFG))


## 2. Подготовка фото: face crop (MTCNN)


In [ ]:
from facenet_pytorch import MTCNN as MTCNNDetector

_detector = MTCNNDetector(keep_all=False, device="cpu", select_largest=True)


def crop_to_face(img: Image.Image, size: int, face_fraction: float) -> tuple[Image.Image, bool]:
    boxes, _ = _detector.detect(img)
    if boxes is not None and len(boxes) > 0:
        x1, y1, x2, y2 = [float(v) for v in boxes[0]]
        face_h = y2 - y1
        crop_size = face_h / face_fraction
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        half = crop_size / 2
        left = max(0, int(cx - half))
        top = max(0, int(cy - half))
        right = min(img.width, int(cx + half))
        bottom = min(img.height, int(cy + half))
        cropped = img.crop((left, top, right, bottom))
        face_found = True
    else:
        s = min(img.size)
        cropped = transforms.CenterCrop(s)(img)
        face_found = False
    return cropped.resize((size, size), Image.LANCZOS), face_found


def list_training_images(folder: Path) -> list[Path]:
    exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    files = sorted(p for p in folder.iterdir() if p.suffix.lower() in exts)
    if len(files) < 3:
        raise FileNotFoundError(f"Нужно минимум 3 фото в {folder}, найдено {len(files)}.")
    return files


photo_paths = list_training_images(PHOTOS_DIR)
face_paths: list[Path] = []
n_no_face = 0

for path in photo_paths:
    img = Image.open(path).convert("RGB")
    face, found = crop_to_face(img, CFG.img_size, CFG.face_fraction)
    out = FACES_DIR / path.name
    face.save(out)
    face_paths.append(out)
    if not found:
        n_no_face += 1

print(f"Исходных: {len(photo_paths)}, face crop: {len(face_paths)}, без лица: {n_no_face}")

n_show = min(4, len(photo_paths))
fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6), squeeze=False)
for i in range(n_show):
    axes[0, i].imshow(Image.open(photo_paths[i]).convert("RGB"))
    axes[0, i].axis("off")
    axes[0, i].set_title("Исходное", fontsize=8)
    axes[1, i].imshow(Image.open(face_paths[i]).convert("RGB"))
    axes[1, i].axis("off")
    axes[1, i].set_title("Face crop", fontsize=8)
plt.suptitle("Тренировочные изображения: до и после face crop", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "face_crop_preview.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Датасет и загрузка модели


In [ ]:
class TextualInversionDataset(Dataset):
    def __init__(self, photo_files: list[Path], tokenizer, templates: list[str], size: int = 512):
        self.photo_files = photo_files
        self.tokenizer = tokenizer
        self.templates = templates
        self.transform = transforms.Compose(
            [
                transforms.Resize(size, interpolation=transforms.InterpolationMode.BILINEAR),
                transforms.CenterCrop(size),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
                transforms.ToTensor(),
                transforms.Normalize([0.5], [0.5]),
            ]
        )

    def __len__(self) -> int:
        return len(self.photo_files) * len(self.templates)

    def __getitem__(self, idx: int):
        img_path = self.photo_files[idx % len(self.photo_files)]
        prompt = random.choice(self.templates).format(PLACEHOLDER_TOKEN)
        pixel = self.transform(Image.open(img_path).convert("RGB"))
        token_ids = self.tokenizer(
            prompt,
            padding="max_length",
            truncation=True,
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt",
        ).input_ids[0]
        return {"pixel_values": pixel, "input_ids": token_ids}


from diffusers import AutoencoderKL, DDPMScheduler, StableDiffusionPipeline, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer

print(f"Загружаем {MODEL_ID} ...")
tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder").to(DEVICE)
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae").to(DEVICE, dtype=WEIGHT_DTYPE)
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet").to(DEVICE, dtype=WEIGHT_DTYPE)
noise_sched = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
print("Компоненты загружены")


## 4. Новый токен и инициализация эмбеддинга


In [ ]:
if PLACEHOLDER_TOKEN not in tokenizer.get_vocab():
    tokenizer.add_tokens([PLACEHOLDER_TOKEN])

placeholder_token_id = tokenizer.convert_tokens_to_ids(PLACEHOLDER_TOKEN)
print(f"Токен {PLACEHOLDER_TOKEN!r}, id = {placeholder_token_id}")

text_encoder.resize_token_embeddings(len(tokenizer))

init_ids = tokenizer.encode(INITIALIZER_TOKEN, add_special_tokens=False)
with torch.no_grad():
    init_embed = text_encoder.get_input_embeddings().weight[init_ids[0]].clone()
    text_encoder.get_input_embeddings().weight[placeholder_token_id] = init_embed

print(f"Эмбеддинг инициализирован из {INITIALIZER_TOKEN!r}")

vae.requires_grad_(False)
unet.requires_grad_(False)
text_encoder.requires_grad_(False)
text_encoder.get_input_embeddings().requires_grad_(True)
print(f"Обучаем 1 строку эмбеддингов ({text_encoder.config.hidden_size} параметров)")


## 5. Обучение Textual Inversion


In [ ]:
dataset = TextualInversionDataset(face_paths, tokenizer, TRAIN_TEMPLATES, size=CFG.img_size)
dataloader = DataLoader(dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=0)

optimizer = torch.optim.AdamW(
    text_encoder.get_input_embeddings().parameters(),
    lr=CFG.learning_rate,
    betas=(0.9, 0.999),
    eps=1e-8,
)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG.train_steps, eta_min=CFG.learning_rate * 0.05
)

frozen_mask = torch.ones(len(tokenizer), dtype=torch.bool)
frozen_mask[placeholder_token_id] = False

loss_history: list[float] = []
lr_history: list[float] = []

if SKIP_TRAIN and EMBEDS_FILE.exists():
    print(f"SKIP_TRAIN=1, пропускаем обучение ({EMBEDS_FILE})")
elif EMBEDS_FILE.exists() and os.environ.get("FORCE_RETRAIN", "0") != "1":
    print(f"Эмбеддинг уже есть: {EMBEDS_FILE} (FORCE_RETRAIN=1 для переобучения)")
else:
    text_encoder.train()
    global_step = 0
    pbar = tqdm(total=CFG.train_steps, desc="Textual Inversion")

    while global_step < CFG.train_steps:
        for batch in dataloader:
            if global_step >= CFG.train_steps:
                break

            pixel_values = batch["pixel_values"].to(DEVICE, dtype=WEIGHT_DTYPE)
            input_ids = batch["input_ids"].to(DEVICE)

            with torch.no_grad():
                latents = vae.encode(pixel_values).latent_dist.sample()
                latents = latents * vae.config.scaling_factor

            noise = torch.randn_like(latents)
            timesteps = torch.randint(
                0, noise_sched.config.num_train_timesteps, (latents.shape[0],), device=DEVICE
            ).long()
            noisy = noise_sched.add_noise(latents, noise, timesteps)

            encoder_hidden = text_encoder(input_ids)[0]
            noise_pred = unet(noisy, timesteps, encoder_hidden.to(WEIGHT_DTYPE)).sample

            loss = F.mse_loss(noise_pred.float(), noise.float())
            loss.backward()

            with torch.no_grad():
                grads = text_encoder.get_input_embeddings().weight.grad
                if grads is not None:
                    grads[frozen_mask] = 0.0

            if global_step == 0:
                g = text_encoder.get_input_embeddings().weight.grad
                assert g is not None and g[placeholder_token_id].abs().sum() > 0, (
                    "Градиент не дошёл до токена — проверьте frozen_mask"
                )
                print(f"  Grad OK: norm = {g[placeholder_token_id].norm():.4f}")

            torch.nn.utils.clip_grad_norm_(text_encoder.get_input_embeddings().parameters(), max_norm=1.0)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

            loss_history.append(loss.item())
            lr_history.append(optimizer.param_groups[0]["lr"])
            global_step += 1
            if global_step % 100 == 0:
                pbar.set_postfix(loss=f"{loss_history[-1]:.4f}", lr=f"{lr_history[-1]:.2e}")
            pbar.update(1)

    pbar.close()
    text_encoder.eval()
    print("Обучение завершено")

    embed = text_encoder.get_input_embeddings().weight[placeholder_token_id].detach().cpu()
    torch.save({PLACEHOLDER_TOKEN: embed}, EMBEDS_FILE)
    print(f"Эмбеддинг сохранён: {EMBEDS_FILE}")

    with open(LOSS_HISTORY_PATH, "w") as f:
        json.dump({"loss": loss_history, "lr": lr_history}, f)

if not EMBEDS_FILE.exists():
    raise FileNotFoundError(f"Нет {EMBEDS_FILE}. Запустите обучение (раздел 5).")

saved = torch.load(EMBEDS_FILE, map_location="cpu", weights_only=True)
with torch.no_grad():
    text_encoder.get_input_embeddings().weight[placeholder_token_id] = saved[PLACEHOLDER_TOKEN].to(DEVICE)
text_encoder.eval()
print(f"Эмбеддинг готов: {EMBEDS_FILE}")

if loss_history or LOSS_HISTORY_PATH.exists():
    hist = loss_history or json.loads(LOSS_HISTORY_PATH.read_text())
    losses = hist["loss"] if isinstance(hist, dict) else hist
    lrs = hist.get("lr", []) if isinstance(hist, dict) else []
    fig, axes = plt.subplots(1, 2 if lrs else 1, figsize=(12 if lrs else 6, 3))
    if not lrs:
        axes = [axes]
    axes[0].plot(losses)
    axes[0].set_title("MSE Loss")
    axes[0].set_xlabel("step")
    if lrs:
        axes[1].plot(lrs)
        axes[1].set_title("Learning Rate")
        axes[1].set_xlabel("step")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
    plt.show()


## 6. Генерация

Pipeline собирается из **тех же** компонентов после обучения (как в эталоне). Без токена в промпте UNet/VAE работают как базовая SD 1.5.


In [ ]:
text_encoder.to(DEVICE, dtype=WEIGHT_DTYPE)

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    text_encoder=text_encoder,
    tokenizer=tokenizer,
    vae=vae,
    unet=unet,
    torch_dtype=WEIGHT_DTYPE,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe = pipe.to(DEVICE)
pipe.set_progress_bar_config(disable=False)
if DEVICE in ("cuda", "mps"):
    pipe.enable_attention_slicing()
pipe.vae.to(dtype=torch.float32)


def generate(prompts: list[str], seed: int = CFG.seed) -> list[Image.Image]:
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    with torch.inference_mode():
        return pipe(
            prompts,
            num_inference_steps=CFG.num_inference_steps,
            guidance_scale=CFG.guidance_scale,
            generator=generator,
        ).images


def show_grid(images: list[Image.Image], titles: list[str], ncols: int = 3, save_prefix: str | None = None):
    n = len(images)
    ncols = min(ncols, n)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    for ax in axes[len(images) :]:
        ax.axis("off")
    plt.tight_layout()
    if save_prefix:
        for i, img in enumerate(images):
            img.save(OUTPUT_DIR / f"{save_prefix}_{i+1}.png")
        plt.savefig(OUTPUT_DIR / f"{save_prefix}_grid.png", dpi=150, bbox_inches="tight")
    plt.show()


### 6.1. Пять изображений в разных стилях


In [ ]:
CREATIVE_PROMPTS = [
    f"a portrait photo of {PLACEHOLDER_TOKEN} in a cyberpunk city, neon lights, looking at the camera, well lit face, high quality",
    f"a portrait photo of {PLACEHOLDER_TOKEN} made of chrome metal, futuristic, looking at the camera, detailed face, studio lighting",
    f"a portrait photo of {PLACEHOLDER_TOKEN} in an elven city, fantasy setting, looking at the camera, detailed face, magical lighting",
    f"a portrait photo of {PLACEHOLDER_TOKEN} as a medieval knight, armor, looking at the camera, detailed face, dramatic lighting",
    f"a portrait photo of {PLACEHOLDER_TOKEN} in space, astronaut, looking at the camera, detailed face, stars background, high quality",
]
CREATIVE_TITLES = ["Cyberpunk", "Chrome metal", "Elven city", "Medieval knight", "Space"]

creative_images = generate(CREATIVE_PROMPTS, seed=CFG.seed)
show_grid(creative_images, CREATIVE_TITLES, ncols=3, save_prefix="creative")


### 6.2. Токен: forest / city / beach


In [ ]:
TOKEN_SCENES = [
    f"a portrait photo of {PLACEHOLDER_TOKEN} in a forest, looking at the camera, well lit face, natural lighting, high quality, realism",
    f"a portrait photo of {PLACEHOLDER_TOKEN} in a city, looking at the camera, well lit face, urban background, high quality, realism",
    f"a portrait photo of {PLACEHOLDER_TOKEN} on a beach, looking at the camera, well lit face, sunny day, high quality, realism",
]
TOKEN_TITLES = [
    f"{PLACEHOLDER_TOKEN} in a forest",
    f"{PLACEHOLDER_TOKEN} in a city",
    f"{PLACEHOLDER_TOKEN} on a beach",
]

token_images = generate(TOKEN_SCENES, seed=CFG.seed + 100)
show_grid(token_images, TOKEN_TITLES, save_prefix="token_env")


### 6.3. Только пол (без токена) — prior preservation


In [ ]:
GENDER_SCENES = [
    f"a portrait photo of a {GENDER} in a forest, looking at the camera, well lit face, natural lighting, high quality, realism",
    f"a portrait photo of a {GENDER} in a city, looking at the camera, well lit face, urban background, high quality, realism",
    f"a portrait photo of a {GENDER} on a beach, looking at the camera, well lit face, sunny day, high quality, realism",
]
GENDER_TITLES = [f"{GENDER} in a forest", f"{GENDER} in a city", f"{GENDER} on a beach"]

print(f"Генерируем '{GENDER}' без токена {PLACEHOLDER_TOKEN}...")
gender_images = generate(GENDER_SCENES, seed=CFG.seed + 200)
show_grid(gender_images, GENDER_TITLES, save_prefix="gender_env")


## 7. Метрики качества (не FID, не IS)

| Метрика | Зачем |
|---------|-------|
| **CLIP Score** | соответствие изображения промпту |
| **Identity Similarity** (FaceNet) | портретное сходство с вашими фото |
| **LPIPS** | перцептуальная близость к референсам |

**Ожидания:** CLIP ~0.25–0.30; Identity Sim для токена 0.5–0.6; для `{GENDER}` без токена — ниже.


In [ ]:
import lpips
import open_clip
from facenet_pytorch import MTCNN, InceptionResnetV1

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
clip_model = clip_model.to(DEVICE).eval()
clip_tok = open_clip.get_tokenizer("ViT-B-32")
lpips_fn = lpips.LPIPS(net="alex").to(DEVICE).eval()
_to_lpips = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ]
)

mtcnn = MTCNN(image_size=160, margin=20, keep_all=False, device="cpu", post_process=False)
facenet = InceptionResnetV1(pretrained="vggface2").eval().to(DEVICE)


@torch.no_grad()
def clip_score(images: list[Image.Image], prompts: list[str]) -> list[float]:
    scores = []
    for img, prompt in zip(images, prompts):
        img_f = clip_model.encode_image(clip_preprocess(img).unsqueeze(0).to(DEVICE))
        txt_f = clip_model.encode_text(clip_tok([prompt]).to(DEVICE))
        img_f = img_f / img_f.norm(dim=-1, keepdim=True)
        txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
        scores.append(float((img_f * txt_f).sum().item()))
    return scores


@torch.no_grad()
def face_embedding(img: Image.Image) -> torch.Tensor | None:
    face = mtcnn(img)
    if face is None:
        return None
    face = ((face.float() / 255.0) - 0.5) / 0.5
    emb = facenet(face.unsqueeze(0).to(DEVICE))
    return emb / emb.norm(dim=-1, keepdim=True)


@torch.no_grad()
def identity_similarity(gen_images: list[Image.Image], ref_images: list[Image.Image]) -> float:
    ref_embeds = [e for img in ref_images if (e := face_embedding(img)) is not None]
    if not ref_embeds:
        return float("nan")
    ref_mean = torch.stack(ref_embeds).mean(0)
    ref_mean = ref_mean / ref_mean.norm()
    sims = [
        float((e * ref_mean).sum().item())
        for img in gen_images
        if (e := face_embedding(img)) is not None
    ]
    return float(np.mean(sims)) if sims else float("nan")


@torch.no_grad()
def mean_lpips(gen_images: list[Image.Image], ref_images: list[Image.Image]) -> float:
    dists = [
        lpips_fn(_to_lpips(g).unsqueeze(0).to(DEVICE), _to_lpips(r).unsqueeze(0).to(DEVICE)).item()
        for g in gen_images
        for r in ref_images
    ]
    return float(np.mean(dists)) if dists else float("nan")


ref_pil = [Image.open(p).convert("RGB") for p in photo_paths]
all_token_images = creative_images + token_images
all_token_prompts = CREATIVE_PROMPTS + TOKEN_SCENES

metrics_summary = {
    "token": {
        "clip_score_mean": float(np.mean(clip_score(all_token_images, all_token_prompts))),
        "lpips_mean": mean_lpips(all_token_images, ref_pil),
        "identity_similarity": identity_similarity(all_token_images, ref_pil),
    },
    "gender_no_token": {
        "clip_score_mean": float(np.mean(clip_score(gender_images, GENDER_SCENES))),
        "lpips_mean": mean_lpips(gender_images, ref_pil),
        "identity_similarity": identity_similarity(gender_images, ref_pil),
    },
}

with open(OUTPUT_DIR / "metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2, ensure_ascii=False)
print(json.dumps(metrics_summary, indent=2, ensure_ascii=False))


## 8. Визуализация метрик


In [ ]:
labels = ["CLIP Score", "LPIPS", "Identity Sim."]
vals_tok = [
    metrics_summary["token"]["clip_score_mean"],
    metrics_summary["token"]["lpips_mean"],
    metrics_summary["token"]["identity_similarity"],
]
vals_gen = [
    metrics_summary["gender_no_token"]["clip_score_mean"],
    metrics_summary["gender_no_token"]["lpips_mean"],
    metrics_summary["gender_no_token"]["identity_similarity"],
]

x, w = np.arange(len(labels)), 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w / 2, vals_tok, w, label=f"Токен ({PLACEHOLDER_TOKEN})")
ax.bar(x + w / 2, vals_gen, w, label=f"Пол ({GENDER})")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.set_title("Персонализированные vs общие промпты")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "metrics_summary.png", dpi=150)
plt.show()


## 9. Выводы

1. **Textual Inversion** обучает только эмбеддинг токена — UNet/VAE неизменны.
2. Промпты с `{PLACEHOLDER_TOKEN}` дают ваше лицо; промпты с `{GENDER}` без токена — произвольного человека.
3. Метрики CLIP / Identity Sim / LPIPS покрывают текст, идентичность и перцептуальное качество.
4. Ограничения: text2img, без negative prompt, без API.
